# 04.1 Multi-Pred Causality

This notebook runs three causality protocols for the multi-predicate setup:

- **4.3a — Nudge Predicate Ordering**: Steer with the P+ direction at the pivot point and check whether the model shifts toward implementing P+ first.
- **4.3b — Adversarial Prefill Persistence**: Load pre-computed results from `scripts/causality/adversarial_persistence.py` and visualise whether P+ directions improve pass rates on adversarial prefills.
- **4.3c — Induce Predicate in Arbitrary Context** *(skeleton)*: The strongest test — steer with P+ direction in a context with no mention of the target predicate.

Prerequisites:
- MCTS has been run and contrastive directions saved by `scripts/multipred_probe.py`.
- For 4.3b: `python -m scripts.causality.adversarial_persistence` has been run first.

## Setup

In [ ]:
import os
import sys
import json
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from rich.console import Console
from rich.table import Table

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src") if "notebooks" in os.getcwd() else ".")
os.chdir(os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else os.getcwd())

console = Console(highlight=False)
rich_print = console.print

rich_print(f"Working directory: {os.getcwd()}")

In [ ]:
from src.models import ModelandTokenizer
from src.goals.data import IncompleteCodeDataset
from src.goals.predicate_data import MultiPredicateDataset, TargetPredicate
from src.utils import env_utils

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
rich_print(f"Model loaded: {mt.name}, n_layer={mt.n_layer}, n_embd={mt.n_embd}")

In [ ]:
# Load datasets
multi_dataset = MultiPredicateDataset.load_dataset()
single_pred_dataset = IncompleteCodeDataset.load_dataset()

rich_print(f"MultiPredicateDataset: {len(multi_dataset)} predicates")
rich_print(f"IncompleteCodeDataset: {len(single_pred_dataset)} samples")
rich_print(f"Available predicates: {[tp.name for tp in multi_dataset.groups]}")

In [ ]:
# ── Pick target predicate + sample ───────────────────────────────────────────
PRED_NAME = multi_dataset.groups[0].name  # Change to pick a different predicate
SAMPLE_IDX = 0

target_predicate: TargetPredicate = next(tp for tp in multi_dataset.groups if tp.name == PRED_NAME)
sample = target_predicate.samples[SAMPLE_IDX]

rich_print(f"Target predicate: {target_predicate.name!r}")
rich_print(f"  P+: {target_predicate.p_plus.description}")
rich_print(f"  P+ anchor tokens: {target_predicate.p_plus.anchor_tokens}")
if sample.p_minus:
    rich_print(f"  P- (sample {SAMPLE_IDX}): {sample.p_minus.description}")
    rich_print(f"  P- anchor tokens: {sample.p_minus.anchor_tokens}")
rich_print(f"\nCode stub:\n{sample.code}")

## 4.3a — Nudge Predicate Ordering

**Protocol**: Steer generation with the P+ direction at the pivot point (right after `return`). Measure whether the steered model shifts toward implementing P+ first, even when given a P--aligned prefill.

**Metric**: Anchor-token scoring — `predicate_order()` checks whether P+ or P- anchor tokens appear first in the generated code. All orderings are correct, so unit-test pass rate is not the metric here.

**Setup**:
- Baseline: natural generation (no steering), with a P--aligned prefill to stress-test
- Steered: same prompt + P+ direction added to hidden states at last position
- Coeff sweep over a small grid × 2-3 layer windows

In [ ]:
# ── 4.3a config ──────────────────────────────────────────────────────────────
PRED_NAME   = target_predicate.name   # re-declare here for easy override
SAMPLE_IDX  = 0

model_short_name = mt.name.split("/")[-1]
SAVE_DIR = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "multipred_probe_test",
    model_short_name,
    PRED_NAME,
)
rich_print(f"Loading directions from: {SAVE_DIR}")

In [ ]:
# ── Load contrastive + mean-diff directions ───────────────────────────────────
contrastive_directions = torch.load(
    os.path.join(SAVE_DIR, "contrastive_directions.pt"),
    map_location="cuda:0",
    weights_only=False,
)
mean_diff_directions = torch.load(
    os.path.join(SAVE_DIR, "mean_diff_directions.pt"),
    map_location="cuda:0",
    weights_only=False,
)
rich_print(f"Contrastive directions loaded: {len(contrastive_directions)} layers")
rich_print(f"Mean-diff directions loaded:   {len(mean_diff_directions)} layers")

In [ ]:
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch, get_hs, PatchSpec
from src.tokens import prepare_input

# ── Build prompt ─────────────────────────────────────────────────────────────
# Use an aligned_p_minus prefill to stress-test (the model wants to do P- first)
aligned_p_minus_prefills = [
    pf for pf in (sample.prefills or []) if pf.get("type") == "aligned_p_minus"
]
if aligned_p_minus_prefills:
    PREFILL_4A = aligned_p_minus_prefills[0]["code"]
    rich_print(f"Using aligned_p_minus prefill: {PREFILL_4A!r}")
else:
    PREFILL_4A = ""  # empty = no nudge
    rich_print("No aligned_p_minus prefill found, using empty prefill")

if supports_chat_template(mt):
    prompt_4a = apply_chat_template(
        mt=mt,
        user_query=sample.instruction.strip(),
        incomplete_code=sample.code + PREFILL_4A,
    )
else:
    prompt_4a = sample.instruction + "<code>" + sample.code + PREFILL_4A

rich_print(f"\nPrompt (last 200 chars): ...{prompt_4a[-200:]!r}")

In [ ]:
# ── Extract hidden states (all layers, last position) ─────────────────────────
tokenized = prepare_input(prompts=prompt_4a, tokenizer=mt)
locations_4a = [(mt.layer_name_format.format(l), -1) for l in range(mt.n_layer)]
hs_4a = get_hs(mt=mt, input=tokenized, locations=locations_4a)
rich_print(f"Hidden states extracted: {len(hs_4a)} locations")

In [ ]:
from scripts.causality.adversarial_persistence import build_steer_patches, generate_and_evaluate

# ── Baseline generation ───────────────────────────────────────────────────────
N_GENERATIONS_4A = 50

baseline_eval_4a = generate_and_evaluate(
    mt=mt,
    sample=sample,
    prefill=PREFILL_4A,
    prompt=prompt_4a,
    n_generations=N_GENERATIONS_4A,
    patches=None,
    label="baseline",
)
baseline_outputs_4a = [gen.lm_generation for gen in baseline_eval_4a.generations]
rich_print(f"Baseline generation done: {len(baseline_outputs_4a)} samples")

In [ ]:
# ── predicate_order helper (verbatim from cell 40 of 03.1_multi_probe.ipynb) ──
from src.goals.anchor import _extract_boolean_expr, _split_clauses

p_plus_anchors  = target_predicate.p_plus.anchor_tokens
p_minus_anchors = sample.p_minus.anchor_tokens if sample.p_minus else []


def predicate_order(text: str) -> str:
    """Return 'p_plus_first', 'p_minus_first', or 'unclear'."""
    full = sample.code + PREFILL_4A + text
    result = _extract_boolean_expr(full)
    if result is None:
        p_plus_pos  = min((full.find(a) for a in p_plus_anchors  if full.find(a) >= 0), default=None)
        p_minus_pos = min((full.find(a) for a in p_minus_anchors if full.find(a) >= 0), default=None)
    else:
        expr, _ = result
        p_plus_pos  = next((expr.find(a) for a in p_plus_anchors  if expr.find(a) >= 0), None)
        p_minus_pos = next((expr.find(a) for a in p_minus_anchors if expr.find(a) >= 0), None)

    if p_plus_pos is None and p_minus_pos is None:
        return "unclear"
    if p_minus_pos is None:
        return "p_plus_first"
    if p_plus_pos is None:
        return "p_minus_first"
    return "p_plus_first" if p_plus_pos <= p_minus_pos else "p_minus_first"


def _rate(counter, key):
    total = sum(counter.values())
    return counter.get(key, 0) / total if total > 0 else 0.0


# Baseline order counts
baseline_orders_4a = Counter(predicate_order(g) for g in baseline_outputs_4a)

rich_print(f"P+ anchors : {p_plus_anchors}")
rich_print(f"P- anchors : {p_minus_anchors}")
rich_print(f"\nBaseline predicate order: {dict(baseline_orders_4a)}")
rich_print(f"  P+ first rate: {_rate(baseline_orders_4a, 'p_plus_first'):.2f}")

In [ ]:
# ── Coeff sweep ───────────────────────────────────────────────────────────────
# Small grid; run one coeff at a time (comment in the loop for full sweep)
COEFF_GRID_4A  = [-0.08, -0.04, 0.0, 0.04, 0.08, 0.12]
LAYER_RANGES_4A = [(30, 40), (40, 50)]  # (layer_from, layer_to)

# Results dict: results_4a[direction_type][layer_tag][coeff] = p_plus_first_rate
results_4a = {"contrastive": {}, "mean_diff": {}}

for layer_from, layer_to in LAYER_RANGES_4A:
    layer_tag = f"{layer_from}-{layer_to}"
    results_4a["contrastive"][layer_tag] = {}
    results_4a["mean_diff"][layer_tag]   = {}

    for coeff in COEFF_GRID_4A:
        for dir_type, directions in [("contrastive", contrastive_directions), ("mean_diff", mean_diff_directions)]:
            patches = build_steer_patches(
                mt=mt,
                directions=directions,
                hs=hs_4a,
                layer_from=layer_from,
                layer_to=layer_to,
                coeff=coeff,
            )
            steered_eval = generate_and_evaluate(
                mt=mt,
                sample=sample,
                prefill=PREFILL_4A,
                prompt=prompt_4a,
                n_generations=N_GENERATIONS_4A,
                patches=patches if coeff != 0.0 else None,
                label=f"{dir_type} layers={layer_tag} coeff={coeff:.2f}",
            )
            steered_outputs = [gen.lm_generation for gen in steered_eval.generations]
            orders = Counter(predicate_order(g) for g in steered_outputs)
            rate = _rate(orders, "p_plus_first")
            results_4a[dir_type][layer_tag][coeff] = rate
            rich_print(f"  [{dir_type}] layers={layer_tag} coeff={coeff:+.2f} → P+ first={rate:.2f}")

rich_print("\nCoeff sweep complete.")

In [ ]:
# ── Visualise: bar chart (baseline vs best steered) ───────────────────────────
baseline_rate_4a = _rate(baseline_orders_4a, "p_plus_first")

# Find best steered rate across all configs for each direction type
best_steered = {}
for dir_type in ("contrastive", "mean_diff"):
    all_rates = [
        (rate, layer_tag, coeff)
        for layer_tag, coeff_dict in results_4a[dir_type].items()
        for coeff, rate in coeff_dict.items()
        if coeff > 0  # positive = toward P+
    ]
    if all_rates:
        best_rate, best_layer, best_coeff = max(all_rates, key=lambda x: x[0])
        best_steered[dir_type] = (best_rate, best_layer, best_coeff)

labels = ["Baseline"] + [
    f"{dt}\nlayers={v[1]}\ncoeff={v[2]:+.2f}"
    for dt, v in best_steered.items()
]
values = [baseline_rate_4a] + [v[0] for v in best_steered.values()]
colors = ["steelblue"] + ["darkorange", "forestgreen"][:len(best_steered)]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel("P+ first rate")
ax.set_title(f"4.3a: Nudge predicate ordering — {PRED_NAME}\nprefill={PREFILL_4A!r}")
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--", label="chance")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.2f}", ha="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise: line plot of P+ first rate vs coeff per layer window ────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, dir_type in zip(axes, ["contrastive", "mean_diff"]):
    for layer_tag, coeff_dict in results_4a[dir_type].items():
        coeffs = sorted(coeff_dict.keys())
        rates  = [coeff_dict[c] for c in coeffs]
        ax.plot(coeffs, rates, marker="o", label=f"layers {layer_tag}")
    ax.axhline(baseline_rate_4a, color="gray", linewidth=1.2, linestyle="--", label="baseline")
    ax.axhline(0.5, color="lightgray", linewidth=0.8, linestyle=":")
    ax.set_xlabel("Steering coefficient")
    ax.set_ylabel("P+ first rate")
    ax.set_title(f"{dir_type}")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)

fig.suptitle(f"4.3a: P+ first rate vs coeff — {PRED_NAME}  (prefill={PREFILL_4A!r})", y=1.02)
plt.tight_layout()
plt.show()

## 4.3b — Adversarial Prefill Persistence

**Protocol**: Take a standard single-predicate task (e.g., `is_palindrome`) where adversarial prefills (e.g., `1 +`, `(3`) confuse the model into failing. Steer with the goal direction trained from the multi-pred setup. Does the direction "remind" the model what it is supposed to do, improving pass rate?

**Prerequisite**: Run `python -m scripts.causality.adversarial_persistence` first. This notebook loads the pre-computed `summary.json`.

```bash
python -m scripts.causality.adversarial_persistence \\
    --model_name Qwen/Qwen2.5-Coder-32B-Instruct \\
    --probe_dir multipred_probe_test \\
    --results_dir causality
```

In [ ]:
from scripts.causality.results import AdversarialPersistenceResults

# ── 4.3b config ──────────────────────────────────────────────────────────────
PRED_NAME_4B   = PRED_NAME          # can override to pick a different predicate
RESULTS_DIR_4B = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "causality", model_short_name, PRED_NAME_4B
)
summary_path = os.path.join(RESULTS_DIR_4B, "summary.json")

rich_print(f"Loading results from: {summary_path}")
r = AdversarialPersistenceResults.load(summary_path)

rich_print(f"Predicate: {r.data['predicate_name']}")
rich_print(f"P+: {r.data['p_plus_description']}")
rich_print(f"N generations: {r.data['n_generations']}")
rich_print(f"Prefills in results: {list(r.data['prefill_results'].keys())}")

In [ ]:
# ── Summary table: per-prefill baseline / best contrastive Δ / best mean_diff Δ ──
table = Table(title=f"4.3b Summary — {PRED_NAME_4B}", show_lines=True)
table.add_column("Prefill",     style="cyan",    no_wrap=True)
table.add_column("Baseline",    style="white",   justify="right")
table.add_column("Contrastive", style="orange1", justify="right")
table.add_column("Δc",          style="orange1", justify="right")
table.add_column("MeanDiff",    style="green",   justify="right")
table.add_column("Δm",          style="green",   justify="right")
table.add_column("Best layer",  style="dim",     justify="right")

for pfx, pfr in r.data["prefill_results"].items():
    baseline = pfr["baseline_pass_rate"]
    bc = pfr.get("best_contrastive")
    bm = pfr.get("best_mean_diff")

    c_rate_str  = f"{bc['pass_rate']:.3f}" if bc else "—"
    delta_c_str = f"{bc['pass_rate'] - baseline:+.3f}" if bc else "—"
    m_rate_str  = f"{bm['pass_rate']:.3f}" if bm else "—"
    delta_m_str = f"{bm['pass_rate'] - baseline:+.3f}" if bm else "—"
    layer_str   = f"{bc['layer_idx'] if bc else '—'}" 

    table.add_row(
        repr(pfx),
        f"{baseline:.3f}",
        c_rate_str, delta_c_str,
        m_rate_str, delta_m_str,
        layer_str,
    )

console.print(table)

In [ ]:
# ── Heatmap: pass_rate[layer][coeff] for contrastive, for a chosen prefill ───
HEATMAP_PREFILL = list(r.data["prefill_results"].keys())[0]  # change as needed
HEATMAP_DIR_TYPE = "contrastive"  # "contrastive" or "mean_diff"

by_layer = r.data["prefill_results"][HEATMAP_PREFILL]["by_layer"]
baseline_rate_hm = r.data["prefill_results"][HEATMAP_PREFILL]["baseline_pass_rate"]

# Collect all layer_idx and coeff keys
layer_keys = sorted(by_layer.keys(), key=int)
by_coeff_key = f"{HEATMAP_DIR_TYPE}_by_coeff"
all_coeffs = sorted(
    {c for ld in by_layer.values() for c in ld.get(by_coeff_key, {})},
    key=float,
)

# Build 2D array: rows = layers, cols = coeffs
matrix = np.full((len(layer_keys), len(all_coeffs)), np.nan)
for row_i, lk in enumerate(layer_keys):
    for col_j, ck in enumerate(all_coeffs):
        val = by_layer[lk].get(by_coeff_key, {}).get(ck)
        if val is not None:
            matrix[row_i, col_j] = val

fig, ax = plt.subplots(figsize=(max(8, len(all_coeffs) * 0.8), max(4, len(layer_keys) * 0.5 + 1)))
im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="pass rate")
ax.set_xticks(range(len(all_coeffs)))
ax.set_xticklabels(all_coeffs, rotation=45, ha="right")
ax.set_yticks(range(len(layer_keys)))
ax.set_yticklabels([f"layer {lk}" for lk in layer_keys])
ax.set_xlabel("Coefficient")
ax.set_ylabel("Layer center")
ax.set_title(
    f"4.3b Heatmap [{HEATMAP_DIR_TYPE}] — {PRED_NAME_4B}\n"
    f"prefill={HEATMAP_PREFILL!r}  (baseline={baseline_rate_hm:.3f})"
)
# Annotate cells
for ri in range(len(layer_keys)):
    for ci in range(len(all_coeffs)):
        val = matrix[ri, ci]
        if not np.isnan(val):
            ax.text(ci, ri, f"{val:.2f}", ha="center", va="center", fontsize=7,
                    color="black" if 0.3 < val < 0.8 else "white")
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart: for all prefills, grouped bars (baseline / contrastive / mean_diff) ──
pfx_list   = list(r.data["prefill_results"].keys())
baseline_vals     = [r.data["prefill_results"][p]["baseline_pass_rate"] for p in pfx_list]
contrastive_vals  = [
    (r.data["prefill_results"][p]["best_contrastive"] or {}).get("pass_rate", float("nan"))
    for p in pfx_list
]
mean_diff_vals    = [
    (r.data["prefill_results"][p]["best_mean_diff"] or {}).get("pass_rate", float("nan"))
    for p in pfx_list
]

x      = np.arange(len(pfx_list))
width  = 0.25
fig, ax = plt.subplots(figsize=(max(8, len(pfx_list) * 1.2), 5))

b1 = ax.bar(x - width, baseline_vals,    width, label="Baseline",    color="steelblue",   edgecolor="white")
b2 = ax.bar(x,          contrastive_vals, width, label="Contrastive", color="darkorange",  edgecolor="white")
b3 = ax.bar(x + width,  mean_diff_vals,   width, label="Mean-diff",   color="forestgreen", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([repr(p) for p in pfx_list], rotation=35, ha="right", fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Pass rate (best steered)")
ax.set_title(f"4.3b: Adversarial Prefill Persistence — {PRED_NAME_4B}")
ax.legend()

for bar_group in [b1, b2, b3]:
    for bar in bar_group:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.2f}",
                    ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.show()

## 4.3c — Induce Predicate in Arbitrary Context *(skeleton)*

**Protocol** (strongest test): Take an arbitrary code generation context that has *no mention of the target predicate in its instructions*. Steer with the P+ goal direction. Check whether the model spontaneously introduces P+-related code (e.g., inserts a palindrome check where none was requested).

**Claim**: If the direction encodes a generalizable intent to "do P+", steering should induce the model to insert P+-related code even in an unrelated context.

**Metric**: Frequency of P+ anchor tokens in steered vs baseline outputs.

In [ ]:
# TODO: pick an arbitrary prompt with no mention of P+ or its function
# Example: a simple list-sorting or string-reversal task
# ARBITRARY_PROMPT = apply_chat_template(mt=mt, user_query="...", incomplete_code="...")
pass

In [ ]:
# TODO: build steering patches (reuse build_steer_patches from 4.3a)
# Use the best (layer_from, layer_to, coeff) found in 4.3a or 4.3b
# patches_4c = build_steer_patches(mt=mt, directions=contrastive_directions, hs=hs_4c, ...)
pass

In [ ]:
# TODO: generate steered outputs; scan for P+ anchor tokens
# baseline_4c = generate_with_patch(mt=mt, inputs=ARBITRARY_PROMPT, patches=None, ...)
# steered_4c  = generate_with_patch(mt=mt, inputs=ARBITRARY_PROMPT, patches=patches_4c, ...)
pass

In [ ]:
# TODO: compare anchor token frequency baseline vs steered
# def has_p_plus_anchor(text: str) -> bool:
#     return any(a in text for a in p_plus_anchors)
# baseline_freq = sum(has_p_plus_anchor(t) for t in baseline_4c) / len(baseline_4c)
# steered_freq  = sum(has_p_plus_anchor(t) for t in steered_4c)  / len(steered_4c)
# rich_print(f"Baseline P+ anchor freq: {baseline_freq:.2f}")
# rich_print(f"Steered  P+ anchor freq: {steered_freq:.2f}")
pass